### MAURICYCLE AI BIN REGISTRATION FORM

This form will allow users to register entries, generate QR codes, and simulate sending PINs via SMS or WhatsApp. The entries will be saved to your provided Excel file.

In [1]:
import sys

!{sys.executable} -m pip install gradio pandas qrcode[pil]

In [43]:
import gradio as gr
import pandas as pd
import qrcode
from PIL import Image
import io
import os
import random # Added for unique PIN generation
from gradio import themes # Import themes for customization

EXCEL_FILE_PATH = '/content/MAURICYCLEAIBINREGISTERFORM.xlsx'

# Define the desired columns for the Gradio form inputs
FORM_INPUT_COLUMNS = [
    "PIN", "SURNAME", "OTHER NAME", "ADDRESS", "TOWN / CITY", "MOBILENUMBER", "NIC NUMBER",
    "DEPENDENT1", "DEPENDENT2", "DEPENDENT3", "DEPENDENT4", "DEPENDENT5"
]

# Define all columns that should be present in the final Excel file
FINAL_EXCEL_COLUMNS = FORM_INPUT_COLUMNS + ["QR_CODE_DATA"]

# Define a mapping from common alternative names (in Excel) to our FORM_INPUT_COLUMNS
# Keys are the potential column names from the Excel file (case-insensitive, space-insensitive normalized)
# Values are the actual FORM_INPUT_COLUMNS names (exact match)
column_name_alternatives = {
    'FIRSTNAME': 'OTHER NAME',
    'MOBILEPHONE': 'MOBILENUMBER',
    'NICNUMBER': 'NIC NUMBER',
    'TOWN/CITY': 'TOWN / CITY',
}

# Define the comprehensive list of Mauritius towns/cities for the dropdown
mauritius_towns_cities = [
    "Port Louis", "Beau Bassin-Rose Hill", "Vacoas-Phoenix", "Curepipe",
    "Quatre Bornes", "Moka", "Pamplemousses", "Flacq", "Grand Port",
    "Rivière Noire", "Plaines Wilhems", "Savanne", "Grand Baie", "Flic en Flac"
]
mauritius_towns_cities.sort()

# Helper function to standardize DataFrame columns
def standardize_dataframe_columns(input_df, expected_cols, alternatives):
    standardized_df = pd.DataFrame(columns=expected_cols)
    for expected_col in expected_cols:
        found_match = False
        if expected_col in input_df.columns:
            standardized_df[expected_col] = input_df[expected_col]
            found_match = True
        else:
            for loaded_col in input_df.columns:
                normalized_loaded_col = loaded_col.replace(' ', '').upper()
                normalized_expected_col = expected_col.replace(' ', '').upper()

                if normalized_loaded_col == normalized_expected_col:
                    standardized_df[expected_col] = input_df[loaded_col]
                    found_match = True
                    break
                if loaded_col.upper() in alternatives and alternatives[loaded_col.upper()] == expected_col:
                    standardized_df[expected_col] = input_df[loaded_col]
                    found_match = True
                    break
        if not found_match:
            standardized_df[expected_col] = pd.NA
    return standardized_df


# Try to load the Excel file
try:
    # Load the existing Excel file
    loaded_df = pd.read_excel(EXCEL_FILE_PATH)

    # Standardize columns of the initially loaded DataFrame using FINAL_EXCEL_COLUMNS
    df = standardize_dataframe_columns(loaded_df, FINAL_EXCEL_COLUMNS, column_name_alternatives)

except FileNotFoundError:
    print(f"Error: Excel file not found at {EXCEL_FILE_PATH}. Creating a dummy file with expected columns.")
    # Use FINAL_EXCEL_COLUMNS for new DataFrame creation
    df = pd.DataFrame(columns=FINAL_EXCEL_COLUMNS)
    df.to_excel(EXCEL_FILE_PATH, index=False)
except Exception as e:
    print(f"An error occurred while reading the Excel file: {e}. Creating a dummy file with expected columns.")
    # Use FINAL_EXCEL_COLUMNS for new DataFrame creation
    df = pd.DataFrame(columns=FINAL_EXCEL_COLUMNS)
    df.to_excel(EXCEL_FILE_PATH, index=False)


# --- Gradio Interface Functions ---

def generate_unique_pin(surname):
    """Generates a unique PIN based on surname initial and 4 digits."""
    if not surname:
        return "" # Return empty if no surname is provided

    # Ensure df is up-to-date with latest saved data for uniqueness check
    try:
        current_df = pd.read_excel(EXCEL_FILE_PATH)
        # Standardize columns of the existing DataFrame before checking for PINs
        standardized_current_df = standardize_dataframe_columns(current_df, FINAL_EXCEL_COLUMNS, column_name_alternatives)
        existing_pins = standardized_current_df['PIN'].dropna().tolist() if 'PIN' in standardized_current_df.columns else []
    except FileNotFoundError:
        existing_pins = []
    except Exception as e:
        print(f"Warning: Could not read Excel for PIN uniqueness check: {e}. Assuming no existing pins.")
        existing_pins = []

    initial = surname[0].upper()

    while True:
        four_digits = str(random.randint(0, 9999)).zfill(4)
        new_pin = f"{initial}{four_digits}"
        if new_pin not in existing_pins:
            return new_pin

def generate_qr_code(pin, surname, other_name, address, town_city, mobile_number, nic_number):
    """Generates a QR code image from the provided registration data."""
    data = f"PIN: {pin}\nName: {surname} {other_name}\nAddress: {address}, {town_city}\nMobile: {mobile_number}\nNIC: {nic_number}"
    qr = qrcode.QRCode(
        version=1,
        error_correction=qrcode.constants.ERROR_CORRECT_L,
        box_size=10,
        border=4,
    )
    qr.add_data(data)
    qr.make(fit=True)

    img = qr.make_image(fill_color="black", back_color="white")

    # Save image to a BytesIO object for Gradio to display
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    buf.seek(0)
    return Image.open(buf)

def sms_pin():
    """Placeholder for sending SMS PIN functionality."""
    return "SMS PIN functionality not yet implemented."

def whatsapp_pin():
    """Placeholder for sending WhatsApp PIN functionality."""
    return "WhatsApp PIN functionality not yet implemented."

def whatsapp_qr_code():
    """Placeholder for sending WhatsApp QR Code functionality."""
    return "WhatsApp QR Code functionality not yet implemented."

def print_pin():
    """Placeholder for printing PIN functionality."""
    return "Printing PIN functionality not yet implemented."

def register_entry(*args):
    """Saves the form data to the Excel file, including generated QR code data string."""
    global df # Use global df to update it

    # Simplified: Create a temporary list for values corresponding to FORM_INPUT_COLUMNS
    form_values_for_excel = list(args[0:7]) + list(args[8:]) # Exclude the radio button (args[7])

    if len(form_values_for_excel) != len(FORM_INPUT_COLUMNS):
        return f"Error: Input count mismatch for Excel. Expected {len(FORM_INPUT_COLUMNS)} inputs, got {len(form_values_for_excel)}."

    # Create a dictionary for the form inputs that go into Excel
    form_entry_data = dict(zip(FORM_INPUT_COLUMNS, form_values_for_excel))

    # Extract data needed for QR code string generation from the form_entry_data dict
    pin = form_entry_data.get("PIN", "")
    surname = form_entry_data.get("SURNAME", "")
    other_name = form_entry_data.get("OTHER NAME", "")
    address = form_entry_data.get("ADDRESS", "")
    town_city = form_entry_data.get("TOWN / CITY", "")
    mobile_number = form_entry_data.get("MOBILENUMBER", "")
    nic_number = form_entry_data.get("NIC NUMBER", "")

    # Generate the QR code data string from the current inputs
    qr_data_string = (
        f"PIN: {pin}\nName: {surname} {other_name}\nAddress: {address}, {town_city}\n"
        f"Mobile: {mobile_number}\nNIC: {nic_number}"
    )

    # Combine form data with QR code data
    full_entry_data = {**form_entry_data, "QR_CODE_DATA": qr_data_string}

    # Create a DataFrame for the new entry, ensuring all FINAL_EXCEL_COLUMNS are present
    new_entry_df = pd.DataFrame([full_entry_data], columns=FINAL_EXCEL_COLUMNS)

    try:
        # Read the existing file again to ensure we have the latest version
        # before appending, in case it was modified externally
        raw_existing_df = pd.read_excel(EXCEL_FILE_PATH)

        # Standardize columns of the existing DataFrame before concatenation
        # Use FINAL_EXCEL_COLUMNS for standardization
        standardized_existing_df = standardize_dataframe_columns(raw_existing_df, FINAL_EXCEL_COLUMNS, column_name_alternatives)

        # Concatenate the standardized existing data with the new entry
        updated_df = pd.concat([standardized_existing_df, new_entry_df], ignore_index=True)
        updated_df.to_excel(EXCEL_FILE_PATH, index=False)
        df = updated_df # Update the global DataFrame
        return "Registration Successful! Data saved to Excel."
    except Exception as e:
        return f"Error saving registration: {e}"

def clear_form():
    """Clears all input fields on the form and resets dependent radio to 'No'."""
    return [
        "", # pin_input
        "", # surname_input
        "", # other_name_input
        "", # address_input
        None, # town_city_dropdown
        "", # mobile_input
        "", # nic_input
        "No", # any_dependent_radio value (reset to No)
        "", # dependent1_input
        "", # dependent2_input
        "", # dependent3_input
        "", # dependent4_input
        "", # dependent5_input
        "", # register_output
        None, # qr_code_output (Image should be None for clearing)
        "", # pin_output
    ]

# Determine the number of dependent fields and create a list for them
dependent_labels = [f'DEPENDENT{i}' for i in range(1, 6)]

# Print the actual list of towns/cities being used for the dropdown
print(f"Towns/Cities loaded for dropdown: {mauritius_towns_cities}")

# Gradio interface layout with custom theme
with gr.Blocks(
    title="MAURICYCLE AI BIN REGISTRATION FORM",
    theme=themes.Default(
        primary_hue="blue",        # Retain a professional blue for primary elements
        secondary_hue="sky",       # Complementary lighter blue
        neutral_hue="gray"        # Clean background for a professional look
    )
) as demo:
    gr.Image(value="/content/newlogo.jpeg", label="Mauricycle Logo", show_label=False, width=100, height=100, interactive=False, show_download_button=False)
    gr.Markdown("<center># MAURICYCLE AI BIN REGISTRATION FORM</center>")

    with gr.Row():
        with gr.Column(scale=1): # Retained 1:2 scale ratio with the right column
            surname_input = gr.Textbox(label="SURNAME")
            other_name_input = gr.Textbox(label="OTHER NAME")
            address_input = gr.Textbox(label="ADDRESS")
            town_city_dropdown = gr.Dropdown(mauritius_towns_cities, label="TOWN / CITY")
            mobile_input = gr.Textbox(label="MOBILENUMBER")
            nic_input = gr.Textbox(label="NIC NUMBER")

            # New: Radio button for 'ANY DEPENDENT?'
            with gr.Row():
                any_dependent_radio = gr.Radio(
                    ["Yes", "No"],
                    label="ANY DEPENDENT?",
                    value="No", # Default to No, dependents hidden
                    interactive=True
                )

            # Dependent fields, initially hidden
            dependent_inputs = [] # List to hold references to dependent textboxes
            for i, dep_label in enumerate(dependent_labels):
                dependent_input = globals()[f'dependent{i+1}_input'] = gr.Textbox(
                    label=dep_label,
                    placeholder=f"Enter {dep_label}'s name (optional)",
                    visible=False # Initially hidden
                )
                dependent_inputs.append(dependent_input)

            register_output = gr.Textbox(label="Registration Status", interactive=False)

            with gr.Row():
                gr.Column(scale=1) # Left spacer for Register/Cancel buttons
                with gr.Column(scale=0): # Make button column take minimum space
                    register_btn = gr.Button("REGISTER")
                with gr.Column(scale=0): # Make button column take minimum space
                    cancel_btn = gr.Button("CANCEL")
                gr.Column(scale=1) # Right spacer for Register/Cancel buttons

        with gr.Column(scale=2): # Retained 1:2 scale ratio with the left column
            # Wrapper to control the width of the PIN textbox and align left
            with gr.Row():
                pin_input = gr.Textbox(label="PIN", interactive=False, scale=2)
                gr.Column(scale=1) # Right spacer to push left

            # Buttons under PIN textbox, stacked vertically and left-aligned
            with gr.Column(): # Column to hold the stacked buttons, will naturally align left
                with gr.Column(scale=0):
                    sms_pin_btn = gr.Button("SMS PIN")
                with gr.Column(scale=0):
                    whatsapp_pin_btn = gr.Button("WApp PIN")
                with gr.Column(scale=0):
                    print_pin_btn = gr.Button("PRINT PIN")

            # QR Code output, now left-aligned (removed centering row)
            qr_code_output = gr.Image(label="SCAN QR CODE", type="pil", width=200, height=200)

            # Buttons for QR codes, stacked vertically and left-aligned
            with gr.Column(): # Column to hold both QR code buttons, naturally stacking them left
                with gr.Column(scale=0):
                    generate_qr_btn = gr.Button("GENERATE QR CODE")
                with gr.Column(scale=0):
                    whatsapp_qr_code_btn = gr.Button("WApp QR CODE") # Moved here, stacked under generate_qr_btn

            # Wrapper to control the width of the PIN Status textbox and align left
            with gr.Row():
                pin_output = gr.Textbox(label="PIN Status", interactive=False, scale=2)
                gr.Column(scale=1) # Right spacer to push left

    # Event Handlers
    # Trigger PIN generation when surname changes
    surname_input.change(generate_unique_pin, inputs=surname_input, outputs=pin_input)

    # Register button
    # all_inputs now includes any_dependent_radio
    all_inputs = (
        [
            pin_input, surname_input, other_name_input, address_input, town_city_dropdown, mobile_input, nic_input,
            any_dependent_radio # Added radio button
        ] +
        dependent_inputs # List of 5 dependent textboxes
    )

    register_btn.click(register_entry, inputs=all_inputs, outputs=register_output)

    # Cancel button
    # Updated outputs to include any_dependent_radio and all dependent_inputs
    cancel_btn.click(
        clear_form,
        inputs=[],
        outputs=(
            [
                pin_input, surname_input, other_name_input, address_input, town_city_dropdown, mobile_input, nic_input,
                any_dependent_radio # Add this to reset its value to "No"
            ] +
            dependent_inputs + # This is the list of 5 dependent textboxes (to clear their values)
            [register_output, qr_code_output, pin_output]
        )
    ).then( # Chain another event to hide dependent fields after clearing
        fn=lambda: [gr.update(visible=False) for _ in range(len(dependent_inputs))],
        inputs=None, # No direct inputs needed for this lambda, as it's triggered after 'clear_form'
        outputs=dependent_inputs
    )

    # Generate QR Code button
    generate_qr_btn.click(generate_qr_code, inputs=[pin_input, surname_input, other_name_input, address_input, town_city_dropdown, mobile_input, nic_input], outputs=qr_code_output)

    # SMS PIN button
    sms_pin_btn.click(sms_pin, inputs=[], outputs=pin_output)

    # WApp PIN button
    whatsapp_pin_btn.click(whatsapp_pin, inputs=[], outputs=pin_output)

    # WApp QR CODE button
    whatsapp_qr_code_btn.click(whatsapp_qr_code, inputs=[], outputs=pin_output)

    # PRINT PIN button
    print_pin_btn.click(print_pin, inputs=[], outputs=pin_output)

    # New: Toggle dependent fields visibility based on radio button
    def toggle_dependent_visibility(choice):
        if choice == "Yes":
            return [gr.update(visible=True) for _ in dependent_inputs]
        else:
            return [gr.update(visible=False) for _ in dependent_inputs]

    any_dependent_radio.change(
        toggle_dependent_visibility,
        inputs=any_dependent_radio,
        outputs=dependent_inputs # All dependent textboxes
    )

# Launch the Gradio interface
demo.launch()

Towns/Cities loaded for dropdown: ['Beau Bassin-Rose Hill', 'Curepipe', 'Flacq', 'Flic en Flac', 'Grand Baie', 'Grand Port', 'Moka', 'Pamplemousses', 'Plaines Wilhems', 'Port Louis', 'Quatre Bornes', 'Rivière Noire', 'Savanne', 'Vacoas-Phoenix']


/tmp/ipykernel_32538/3649718463.py:227: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3a8df7c8ed2e5e74c7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [48]:
css_code = """
body .gradio-app {
    max-width: 80%; /* Reduce the overall width of the app to 80% */
    margin: auto; /* Center the app on the page */
}
.gradio-container button {
    padding: 2px 5px !important; /* Reduce padding */
    font-size: 0.8em !important; /* Reduce font size */
    min-height: 20px !important; /* Set minimum height to reduce length */
    width: 120px !important; /* Set uniform width for all buttons */
}
.qr-code-image-container {
    text-align: left !important;
}
"""

# Re-launch the Gradio interface with custom CSS
with gr.Blocks(
    title="MAURICYCLE AI BIN REGISTRATION FORM",
    theme=themes.Default(
        primary_hue="blue",
        secondary_hue="sky",
        neutral_hue="gray"
    ),
    css=css_code # Inject custom CSS
) as demo_css:
    gr.Image(value="/content/newlogo.jpeg", label="Mauricycle Logo", show_label=False, width=100, height=100, interactive=False, show_download_button=False)
    gr.Markdown("<center># MAURICYCLE AI BIN REGISTRATION FORM</center>")

    with gr.Row():
        with gr.Column(scale=1):
            surname_input = gr.Textbox(label="SURNAME")
            other_name_input = gr.Textbox(label="OTHER NAME")
            address_input = gr.Textbox(label="ADDRESS")
            town_city_dropdown = gr.Dropdown(mauritius_towns_cities, label="TOWN / CITY")
            mobile_input = gr.Textbox(label="MOBILENUMBER")
            nic_input = gr.Textbox(label="NIC NUMBER")

            with gr.Row():
                any_dependent_radio = gr.Radio(
                    ["Yes", "No"],
                    label="ANY DEPENDENT?",
                    value="No",
                    interactive=True
                )

            dependent_inputs = []
            for i, dep_label in enumerate(dependent_labels):
                dependent_input = globals()[f'dependent{i+1}_input'] = gr.Textbox(
                    label=dep_label,
                    placeholder=f"Enter {dep_label}'s name (optional)",
                    visible=False
                )
                dependent_inputs.append(dependent_input)

            register_output = gr.Textbox(label="Registration Status", interactive=False)

            with gr.Row():
                gr.Column(scale=1)
                with gr.Column(scale=0):
                    register_btn = gr.Button("REGISTER")
                with gr.Column(scale=0):
                    cancel_btn = gr.Button("CANCEL")
                gr.Column(scale=1)

        with gr.Column(scale=2):
            with gr.Row():
                pin_input = gr.Textbox(label="PIN", interactive=False, scale=2)
                gr.Column(scale=1)

            with gr.Column():
                with gr.Column(scale=0):
                    sms_pin_btn = gr.Button("SMS PIN")
                with gr.Column(scale=0):
                    whatsapp_pin_btn = gr.Button("WApp PIN")
                with gr.Column(scale=0):
                    print_pin_btn = gr.Button("PRINT PIN")

            qr_code_output = gr.Image(label="SCAN QR CODE", type="pil", width=200, height=200, elem_classes="qr-code-image-container")

            with gr.Column():
                with gr.Column(scale=0):
                    generate_qr_btn = gr.Button("GENERATE QR CODE")
                with gr.Column(scale=0):
                    whatsapp_qr_code_btn = gr.Button("WApp QR CODE")

            with gr.Row():
                pin_output = gr.Textbox(label="PIN Status", interactive=False, scale=2)
                gr.Column(scale=1)

    surname_input.change(generate_unique_pin, inputs=surname_input, outputs=pin_input)

    all_inputs = (
        [
            pin_input, surname_input, other_name_input, address_input, town_city_dropdown, mobile_input, nic_input,
            any_dependent_radio
        ] +
        dependent_inputs
    )

    register_btn.click(register_entry, inputs=all_inputs, outputs=register_output)

    cancel_btn.click(
        clear_form,
        inputs=[],
        outputs=(
            [
                pin_input, surname_input, other_name_input, address_input, town_city_dropdown, mobile_input, nic_input,
                any_dependent_radio
            ] +
            dependent_inputs +
            [register_output, qr_code_output, pin_output]
        )
    ).then(
        fn=lambda: [gr.update(visible=False) for _ in range(len(dependent_inputs))],
        inputs=None,
        outputs=dependent_inputs
    )

    generate_qr_btn.click(generate_qr_code, inputs=[pin_input, surname_input, other_name_input, address_input, town_city_dropdown, mobile_input, nic_input], outputs=qr_code_output)

    sms_pin_btn.click(sms_pin, inputs=[], outputs=pin_output)
    whatsapp_pin_btn.click(whatsapp_pin, inputs=[], outputs=pin_output)
    whatsapp_qr_code_btn.click(whatsapp_qr_code, inputs=[], outputs=pin_output)
    print_pin_btn.click(print_pin, inputs=[], outputs=pin_output)

    def toggle_dependent_visibility(choice):
        if choice == "Yes":
            return [gr.update(visible=True) for _ in dependent_inputs]
        else:
            return [gr.update(visible=False) for _ in dependent_inputs]

    any_dependent_radio.change(
        toggle_dependent_visibility,
        inputs=any_dependent_radio,
        outputs=dependent_inputs
    )

demo_css.launch()

/tmp/ipykernel_32538/2000544504.py:15: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_32538/2000544504.py:15: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e3c3d76ee0c3c55d6e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
